In [4]:
import numpy as np
import pandas as pd

# 1. データのインポート（一番左の列をインデックスとして指定）
filename = r"F:\thesis\0616SUNNING-akane-2026-06-16\videos\ワオキツネザルCAM1-20250427-085828-1745711908801-7_00-18-50_00-19-20DLC_resnet50_0616SUNNINGJun16shuffle1_100000.csv"
df = pd.read_csv(filename, header=[0, 1, 2], index_col=0)

# 最上位階層（スコアラー名）の文字列を自動抽出
scorer = df.columns[0][0]

# 2. 尤度（Likelihood）による品質フィルタリング
parts = ["left_hand", "right_hand", "left_ear", "right_ear"]
valid_mask = True
for part in parts:
    valid_mask &= df[(scorer, part, "likelihood")] >= 0.9

# 3. left_hand の X, Y 座標を抽出
hand_x = df[(scorer, "left_hand", "x")]
hand_y = df[(scorer, "left_hand", "y")]

# 4. ローリングウィンドウ（移動窓）を用いた静止度（標準偏差）の計算
window_size = 11  # 前後5フレーム分を含めた窓
std_x = hand_x.rolling(window=window_size, center=True).std()
std_y = hand_y.rolling(window=window_size, center=True).std()

# 総変動量（Motion Index）を算出
motion_index = np.sqrt(std_x**2 + std_y**2)

# 5. サニング（静止状態）の自動判定
# 変動量が 2.0 ピクセル以下を一旦の基準とします
threshold_px = 2.0
is_sunning = (motion_index < threshold_px) & valid_mask

# 6. 判定されたフレームを抽出
sunning_frames = is_sunning[is_sunning == True].index.tolist()

print(f"抽出されたスコアラー名: {scorer}")
print(f"サニング検知フレーム数: {len(sunning_frames)}")

抽出されたスコアラー名: DLC_resnet50_0616SUNNINGJun16shuffle1_100000
サニング検知フレーム数: 143
